In [ ]:
from itertools import product
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
from qiskit_addon_sqd.counts import counts_to_arrays, bit_array_to_arrays,BitArray
from tqdm.notebook import tqdm
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
import os
import matplotlib.pyplot as plt
from glob import glob
import seaborn as sns 
from tqdm.notebook import tqdm
from matplotlib import font_manager
import scipy.stats as stats
from shutil import move
font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 12})

from qiskit_addon_sqd.subsampling import postselect_by_hamming_right_and_left


In [ ]:
datadf = pd.read_csv("../../../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';').dropna(axis=1)

In [ ]:
UofT_palette = [ "#1E3765",
                 "#007FA3", 
                 "#6D247A", 
                 "#DC4633",
                 "#6FC7EA",
                 "#00A189",
                 "#AB1368",
                 "#0D534D",
                 "#F1C500",
                 "#8DBF2E"
               ]

palette = sns.color_palette(UofT_palette)

In [ ]:
import math


def fci_dimension(n_alpha: int, n_beta: int, n_orbitals: int) -> int:
    """Calculates the FCI space dimension for a given number of alpha/beta

    electrons and spatial orbitals.
    """
    alpha_combs = math.comb(n_orbitals, n_alpha)
    beta_combs = math.comb(n_orbitals, n_beta)

    return alpha_combs * beta_combs

In [ ]:
dim_data = []
for i in sorted(glob("./counts/*npz")):
    parts = os.path.basename(i).replace('.npz', '').split('_')
    name, _, rawlayers, basis = parts[:4]
    injection = '_'.join(parts[4:])

    n_elec = datadf.loc[datadf['molecule'] == name, 'Ne'].values[0]
    n_orb = datadf.loc[datadf['molecule'] == name, 'No'].values[0]
    right = left = n_elec // 2
    layers = int(rawlayers.strip("L"))
    exact_dim = fci_dimension(left,right,n_orb)
    counts = np.load(i)
    probarr = counts['probarr']
    bitstrings = counts['bitstrings']  # already the right shape/dtype, no conversion needed

    ps_bitstrings, ps_probs = postselect_by_hamming_right_and_left(
        bitstrings, probarr,
        hamming_right=right, hamming_left=left,
    )
    
    dim_data.append((name, layers, basis, injection, ps_bitstrings.shape[0],bitstrings.shape[0],exact_dim,10000,n_orb,n_elec))

dim_df = pd.DataFrame(dim_data,columns=['Name','L','Basis','Injection','Postselected','Device','Exact','Shots','NOrb','Nelec'])
dim_df.to_excel("Dimensions.xlsx")

In [ ]:
dim_df['Basis']

In [ ]:
n_atoms_dict = {
    'water': 3,
    'methane': 5,
    'ammonia': 4,
    'ethane': 8,
    'methanol': 6,
    'ethylene': 6,
    'formaldehyde': 4,
    "prop-2-en-1-ol":10,
    "but-1-yne":10,
    "fluoroform":5,
    "buta-1,3-diene":10,
    "(Z)-1-fluoroprop-1-ene":9
}

dim_df['n_atoms'] = dim_df['Name'].map(n_atoms_dict)

dim_df['PostselectedPercent'] = (dim_df['Postselected'] / dim_df['Shots'])*1e2

In [ ]:
dim_df

In [ ]:
import matplotlib.patches as mpatches

L_values = sorted(dim_df['L'].unique())
palette_dict = dict(zip(L_values, sns.color_palette(palette, n_colors=len(L_values))))

g = sns.catplot(
    data=dim_df.sort_values(by=['Basis', 'L', 'n_atoms']),
    x="NOrb",
    y="Postselected",
    hue="L",
    col="Basis",
    row='Injection',
    hue_order=L_values,
    col_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],
    whis=(0, 100),
    palette=palette_dict,
    kind="box",
    legend=False,   # suppress the buggy auto legend
)
g.set_axis_labels("Number of Correlated Orbitals", "Number of Unique Configurations")
g.set_xticklabels(rotation=90)

# Build legend manually from the palette dict — guaranteed correct swatches
handles = [mpatches.Patch(facecolor=palette_dict[l], label=str(l)) for l in L_values]
g.figure.legend(
    handles=handles,
    title="Layers",
    bbox_to_anchor=(1.0, 0.5),
    loc="center left",
    frameon=False,
)

g.figure.subplots_adjust(right=0.93)
plt.tight_layout()
plt.savefig("./GMJ_figures/PostselectedConfigVsActiveOrbitals.png", dpi=300, bbox_inches='tight')

In [ ]:
print(len(palette), sorted(dim_df['L'].unique()))

In [ ]:
# sns.scatterplot(merged_df,x='L',y='PostselectedPercent',hue='Basis')
sns.boxplot(
    data=dim_df.sort_values(by=['Basis','L','n_atoms']), x="L", y="Device", hue="Basis",hue_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],whis=(0, 100),palette=palette[0:3]
)

In [ ]:
dim_df['Device']
'Postselected'
'L'

np.unique(dim_df['L']),['STO-3G','cc-pVDZ','aug-cc-pVDZ']
    x=,
    y=,
    hue=,
    hue_order=L_values,
    palette=palette_dict,
    jitter=True,
    dodge=True,
)
ax.set_ylabel("Number of Configurations Postselected")
ax.set_xlabel(r"Number of Unique Configurations from $ibm\_quebec$")

In [ ]:

g = sns.catplot(dim_df.sort_values(by=["Basis","n_atoms","L"]),x='Name',y='PostselectedPercent',hue='L',row='Injection',col='Basis',kind='bar',col_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette,row_order=['zeroes','random','MP2','ML','ML_exact'])
